In [30]:
import h5py
import numpy as np
fpath = "Gmovi.h5"
with h5py.File(fpath, "r") as h5f:
    stats = {}
    for ds in ["poses","trans", "betas"]:
        raw_data = []
        for key in list(h5f["train"].keys()):
            ex = h5f["train"][key]
            raw_data.append(ex[ds])
        if ds == "betas":
            np_data = np.stack(raw_data,axis = 0)
        else:
            np_data = np.concat(raw_data,axis = 0)
        print(np_data.shape)
        stats[ds] = {
            # "data" : np_data,
            "shape" : np_data.shape,
            "mu" : np_data.mean(axis=0).tolist(),
            "sigma" : np_data.std(axis=0).tolist()
        }

            

(967016, 52, 3)
(967016, 3)
(1424, 16)


In [31]:
import json 
with open("normalization.json", "w") as file: 
    json.dump(stats,file)

In [28]:
for ds in ["poses","trans","betas"]:
    print(ds)
    print(stats[ds]["shape"])
    print(stats[ds]["mu"].shape)
    print(stats[ds]["sigma"].shape)
    # print(stats[ds]["mu"])
    # print(stats[ds]["sigma"])
    print()
    print()

poses
(967016, 52, 3)
(52, 3)
(52, 3)


trans
(967016, 3)
(3,)
(3,)


betas
(1424, 16)
(16,)
(16,)




In [ ]:
def upsample(poses, trans, factor=4):
    """
    Upsample `poses` (N, J, C) in time by `factor` (or to match `trans` length).
    Returns (poses_up, trans_up) with time dimension = T.

    - poses: shape (N, J, C)  e.g. (T/4, 52, 3)
    - trans: shape (M, 3)     e.g. (T, 3)
    - factor: int or None. If provided it expects M ~= N*factor. If None, factor is inferred.
    """
    poses = np.asarray(poses)
    trans = np.asarray(trans)

    n_small = poses.shape[0]
    n_trans = trans.shape[0]

    # infer/validate factor and final length
    if factor is None:
        factor = float(n_trans) / float(n_small)
    if abs(round(factor) - factor) < 1e-8:
        factor = int(round(factor))
    n_large = int(round(n_small * factor))

    # target time axis (0 .. n_large-1)
    t_large = np.arange(n_large)
    # small frames spread over full target range so first maps to 0 and last to n_large-1
    t_small = np.linspace(0, n_large - 1, n_small)

    # flatten poses to (n_small, D) and interp each column
    D = poses.shape[1] * poses.shape[2]
    flat = poses.reshape(n_small, D)
    up_flat = np.empty((n_large, D), dtype=flat.dtype)
    for d in range(D):
        up_flat[:, d] = np.interp(t_large, t_small, flat[:, d])
    poses_up = up_flat.reshape(n_large, poses.shape[1], poses.shape[2])

    # resample trans if needed (use its own original time axis)
    if n_trans != n_large:
        t_trans_small = np.linspace(0, n_large - 1, n_trans)
        trans_up = np.empty((n_large, trans.shape[1]), dtype=trans.dtype)
        for c in range(trans.shape[1]):
            trans_up[:, c] = np.interp(t_large, t_trans_small, trans[:, c])
    else:
        trans_up = trans.copy()

    return poses_up, trans_up

In [ ]:
import numpy as np

def upsample(poses, trans, T):
    """
    Upsample `poses` and `trans` along time to length T.

    Args:
        poses: array-like, shape (t0, J, C)  (e.g., (T/4, 52, 3))
        trans: array-like, shape (t0, D) (e.g., (T/4, 3)). Must have same time dimension as poses.
        T: int target time length. 

    Returns:
        poses_up: ndarray shape (T, J, C)
        trans_up: ndarray shape (T, D) or None if trans was None
    """
    poses = np.asarray(poses)
    trans = np.asarray(trans)
    if poses.ndim != 3:
        raise ValueError("poses must have shape (t, J, C)")
    if trans.ndim != 3:
        raise ValueError("trans must have shape (t, C)")
    if poses.shape[0] != trans.shape[0]:
        raise ValueError("poses and trans must have the same time dimension")

    t0 = poses.shape[0]

    if T == t0:
        poses_up = poses.copy()
        trans_up = trans.copy()
    else:
        t_target = np.arange(T)
        t_src = np.linspace(0, T - 1, t0)

        D = poses.shape[1] * poses.shape[2]
        flat = poses.reshape(t0, D).astype(float)
        up_flat = np.empty((T, D), dtype=flat.dtype)
        for d in range(D):
            up_flat[:, d] = np.interp(t_target, t_src, flat[:, d])
        poses_up = up_flat.reshape(T, poses.shape[1], poses.shape[2])

        trans_up = np.empty((T, trans.shape[1]), dtype=trans.dtype)
        for c in range(trans.shape[1]):
            trans_up[:, c] = np.interp(t_target, t_src, trans[:, c])

    return poses_up, trans_up

In [ ]:
"""
movi.h5
    ├── train/
    │   ├── Subject_1__walking/
    │   │   ├── poses   (T, 52, 3)
    │   │   ├── trans   (T,  3)
    │   │   ├── betas   (16,)
    │   │   └── attrs:  gender, action, subject, height, mass, age,
    │   │               framerate, n_frames, split
    │   └── ...
    ├── val/  ...
    └── test/ ...
"""
"""
lifted.h5
    ├── train/
    │   ├── Subject_1__walking/
    |   |   ├── PG1/     
    |   |   |   ├── poses    (T/4,52,3)
    |   |   |   ├── trans    (T/4,3)
    |   |   |   ├── betas    (16,)
    |   |   ├── PG2/     
    |   |   |   ├── poses    (T/4,52,3)
    |   |   |   ├── trans    (T/4,3)
    |   |   |   ├── betas    (16,)
    │   └── ...
    ├── val/  ...
    └── test/ ...
"""

"""
final_dataset.h5
    ├── train/
    │   ├── Subject_1__walking/
    │   │   ├── norm_poses   (T, 52, 3)
    │   │   ├── norm_trans   (T,  3)
    │   │   ├── norm_betas   (16,)
    |   |   ├── norm_PG1/     
    |   |   |   ├── norm_poses    (T,52,3)
    |   |   |   ├── norm_trans    (T,3)
    |   |   |   ├── norm_betas    (16,)
    |   |   ├── norm_PG2/     
    |   |   |   ├── norm_poses    (T,52,3)
    |   |   |   ├── norm_trans    (T,3)
    |   |   |   ├── norm_betas    (16,)
    │   │   └── attrs:  gender, action, subject, height, mass, age,
    │   │               framerate, n_frames, split
    │   └── ...
    ├── val/  ...
    └── test/ ...
"""

In [33]:
import h5py
import numpy as np
fpath = "Gmovi.h5"
split_index = {}
with h5py.File(fpath, "r") as h5f:
    for split in ["train", "val", "test"]:
        split_index[split] = list(h5f[split].keys())

with open("split_index.json", "w") as file:
    json.dump(split_index, file)

In [43]:
import h5py
file = h5py.File("Gmovi.h5", "r")
grp_movi = file["train"]
grp_movi["Subject_1__walking"].attrs.get("n_frames")
# grp_movi["Subject_1__walking"]["poses"].shape
for key, val in grp_movi["Subject_1__walking"].attrs.items():
    print(f"{key}: {val}")

action: walking
age: 25
framerate: 120
gender: male
height: 184
mass: 92
n_frames: 705
split: train
subject: Subject_1
